# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 1: Łańcuchy Markowa (poziom znaków)

W tym zeszycie budujemy prosty model językowy oparty na łańcuchach Markowa, który generuje tekst znak po znaku na podstawie częstości występowania par znaków w tekście.

In [ ]:
import random
from collections import Counter
import numpy as np

In [ ]:
# Wczytujemy tekst z pliku
with open('data/pan-tadeusz.txt', 'r', encoding='utf-8') as f:
    content = f.read()

print(f"Liczba znaków w tekście: {len(content)}")

In [ ]:
# Zliczamy wystąpienia każdego znaku w tekście
counter = Counter(content)
print(f"Liczba unikalnych znaków: {len(counter)}")

# Tworzymy listę unikalnych znaków i słownik mapujący znak na jego indeks
chars = list(counter.keys())
char2id = {c: i for i, c in enumerate(chars)}

In [ ]:
# Inicjalizujemy macierz przejść (liczba wystąpień pary znaków)
n_chars = len(chars)
pairs = np.zeros((n_chars, n_chars))
count = np.zeros(n_chars)

# Zliczamy przejścia z jednego znaku (c1) do następnego (c2)
for i in range(len(content) - 1):
    c1 = content[i]
    c2 = content[i + 1]
    id1 = char2id[c1]
    id2 = char2id[c2]
    
    pairs[id1, id2] += 1
    count[id1] += 1

In [ ]:
def generuj_tekst(znak_poczatkowy: str, dlugosc: int = 1000) -> str:
    """
    Generuje tekst używając łańcucha Markowa.
    
    :param znak_poczatkowy: Znak, od którego zaczynamy generowanie tekstu.
    :param dlugosc: Liczba znaków do wygenerowania.
    :return: Wygenerowany ciąg znaków.
    """
    if znak_poczatkowy not in char2id:
        raise ValueError(f"Znak '{znak_poczatkowy}' nie występuje w tekście.")
        
    aktualny_znak = znak_poczatkowy
    wynik = [aktualny_znak]
    
    for _ in range(dlugosc):
        id_aktualnego = char2id[aktualny_znak]
        
        # Obliczamy prawdopodobieństwa przejścia do kolejnych znaków
        # Dzielimy liczbę wystąpień par przez całkowitą liczbę wystąpień aktualnego znaku
        # Zabezpieczenie przed dzieleniem przez zero (np. gdy znak występuje tylko na końcu tekstu)
        if count[id_aktualnego] == 0:
            break
            
        prawdopodobienstwa = pairs[id_aktualnego] / count[id_aktualnego]
        
        # Losujemy kolejny znak na podstawie wag (prawdopodobieństw)
        aktualny_znak = random.choices(chars, weights=prawdopodobienstwa)[0]
        wynik.append(aktualny_znak)
        
    return "".join(wynik)

In [ ]:
# Generujemy przykładowy tekst zaczynając od litery 'T'
wygenerowany_tekst = generuj_tekst('T', dlugosc=500)
print(wygenerowany_tekst)

---
## Większy rozmiar stanu (n-gramy znakowe)
Powyższe podejście z macierzą numpy jest świetne dla pojedynczych znaków. Aby łatwo uwzględnić dłuższą historię (np. 3 poprzednie znaki), użyjemy podejścia słownikowego, podobnie jak w przypadku słów.

In [ ]:
from collections import defaultdict

def zbuduj_model_znakowy(tekst: str, rozmiar_stanu: int = 2) -> dict[str, Counter]:
    """
    Tworzy model Markowa dla znaków o zadanym rozmiarze stanu.
    
    :param tekst: Tekst wejściowy.
    :param rozmiar_stanu: Liczba znaków brana pod uwagę przy przewidywaniu następnego.
    :return: Słownik mapujący stan (ciąg znaków) na licznik kolejnych znaków.
    """
    model = defaultdict(Counter)
    for i in range(len(tekst) - rozmiar_stanu):
        stan = tekst[i:i+rozmiar_stanu]
        nastepny_znak = tekst[i+rozmiar_stanu]
        model[stan][nastepny_znak] += 1
    return model

# Budujemy model dla 3 znaków wstecz
rozmiar_stanu = 3
model_ngram = zbuduj_model_znakowy(content, rozmiar_stanu=rozmiar_stanu)
print(f"Liczba unikalnych stanów (rozmiar {rozmiar_stanu}): {len(model_ngram)}")

In [ ]:
def generuj_tekst_ngram(model: dict[str, Counter], stan_poczatkowy: str | None = None, dlugosc: int = 500) -> str:
    """
    Generuje tekst na podstawie znakowego modelu n-gramowego.
    """
    if stan_poczatkowy is None:
        stan_poczatkowy = random.choice(list(model.keys()))
        
    aktualny_stan = stan_poczatkowy
    wynik = list(aktualny_stan)
    
    for _ in range(dlugosc):
        if aktualny_stan not in model:
            break
            
        wybory = list(model[aktualny_stan].keys())
        wagi = list(model[aktualny_stan].values())
        
        nastepny_znak = random.choices(wybory, weights=wagi)[0]
        wynik.append(nastepny_znak)
        
        # Aktualizujemy stan - bierzemy ostatnie 'n' znaków
        aktualny_stan = "".join(wynik[-len(aktualny_stan):])
        
    return "".join(wynik)

# Generujemy tekst z modelu 3-znakowego
print(generuj_tekst_ngram(model_ngram, dlugosc=500))